# Lab 2 — ผู้ช่วยตัวเดิมด้วย LangChain (EP03)

ใน Lab 1 เราเขียน harness เองทุกบรรทัด: tool schema, `run_tool`, ลูป, `history`, `MAX_STEPS`, `NEEDS_APPROVAL`
lab นี้สร้าง **ผู้ช่วยวิชา AI301 ตัวเดิม** ด้วย **LangChain** (`create_agent`) แล้วจับคู่ทีละชิ้นกับโค้ดของเรา

| LangChain | ตรงกับที่เราเขียนเองใน Lab 1 |
|---|---|
| `ChatOpenAI(...)` | `client` + `MODEL` + `EXTRA` + `call_llm` |
| `@tool` + type hint + docstring | ฟังก์ชัน + JSON schema ใน `TOOLS` + `FUNCS` + `run_tool` |
| `create_agent(...)` | `run_agent()` — ลูป รับรู้ → คิด → ลงมือ |
| `checkpointer` + `thread_id` | list `history` |
| `ModelCallLimitMiddleware` | `MAX_STEPS` |
| `HumanInTheLoopMiddleware` + `Command(resume=...)` | `NEEDS_APPROVAL` + `PENDING` + `approve()` |

ปฏิทินและห้องติวเป็น **ข้อมูลจำลองชุดเดียวกับ Lab 1** ส่วนคำตอบของ model ทั้งหมดมาจากการรันจริง

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, ch15 และ Table 18.1 — CC BY-SA 4.0

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **6 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep03-lab2-langchain-agent.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## 0. เตรียมเครื่อง

ต้องมี Python 3.10+ (หรือ Colab) และ **OpenRouter API key** — วิธีตั้งค่า key และเหตุผลที่เลือก model อยู่ใน Lab 1

เวอร์ชันของ LangChain ถูกล็อกไว้ เพราะ API ของ framework เปลี่ยนเร็ว ถ้าใช้เวอร์ชันใหม่กว่านี้ บางบรรทัดอาจต้องปรับ

In [ ]:
%pip install -q "langchain==1.4.7" "langgraph==1.2.16" "langchain-openai==1.7.2" "langchain-core==1.7.0" python-dotenv

In [ ]:
import os

# ---- จาก Lab 1: โหลด API key (Colab Secrets หรือไฟล์ .env) ----
def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

## 1. Model: `ChatOpenAI`

ใน Lab 1: `client` + `MODEL` + `EXTRA` และฟังก์ชัน `call_llm` → ใน LangChain รวมเป็น object เดียว

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="google/gemma-4-26b-a4b-it", base_url="https://openrouter.ai/api/v1", api_key=load_key(),
                 extra_body={"reasoning": {"enabled": False},
                             "provider": {"order": ["deepinfra/fp8"], "allow_fallbacks": False}})

## 2. Tool: `@tool`

ใน Lab 1 เราเขียนแต่ละ tool สองครั้ง — ฟังก์ชัน Python และ JSON schema ใน `TOOLS` — แล้วผูกด้วย `FUNCS` + `run_tool`
ใน LangChain เขียนครั้งเดียว: `@tool` อ่าน **ชื่อฟังก์ชัน · type hint · docstring** แล้วสร้าง schema ให้

In [ ]:
# ---- จาก Lab 1: ข้อมูลจำลองเพื่อการสอน (วันนี้ = 2026-11-10) ----
CALENDAR = {"today": "2026-11-10 (วันอังคาร)",
            "events": [{"name": "สอบกลางภาค AI301", "date": "2026-11-18 (วันพุธ)"},
                       {"name": "ส่งการบ้าน 3", "date": "2026-11-14 (วันเสาร์)"}]}
FREE = {"2026-11-12": {"ห้อง 1": ["09:00"], "ห้อง 3": ["13:00", "15:00"], "ห้อง 5": []},
        "2026-11-13": {"ห้อง 1": ["10:00", "13:00"], "ห้อง 3": [], "ห้อง 5": ["15:00"]}}
BOOKING = {}
SYSTEM = "คุณคือผู้ช่วยของวิชา AI301 ช่วยนักศึกษาดูปฏิทินและจองห้องติว ใช้ tool เพื่อหาข้อมูลจริง ห้ามเดาวันที่"
QUESTION = "อีกกี่วันจะสอบกลางภาควิชา AI301 แล้วช่วยจองห้องติววันพฤหัสนี้ช่วงบ่ายให้หน่อย"

In [ ]:
# ✏️ TODO 1 (▶ วิดีโอ 3:38): แปลงฟังก์ชันธรรมดาเป็น tool ที่ LangChain ส่งให้ model ได้
from langchain_core.tools import tool

___
def get_calendar() -> dict:
    """ดูวันที่วันนี้ และปฏิทินสอบ/กำหนดส่งของวิชา AI301"""
    return CALENDAR

@tool
def find_rooms(date: str) -> dict:
    """ดูห้องติวที่ว่างในวันที่กำหนด (date รูปแบบ YYYY-MM-DD)"""
    if date not in FREE:
        return {"error": f"ไม่มีข้อมูลห้องวันที่ {date} (มีข้อมูล: {', '.join(FREE)})"}
    return {"date": date, "free": {r: t for r, t in FREE[date].items() if t}}

@tool
def book_room(room: str, date: str, time: str) -> dict:
    """จองห้องติว (จองได้ครั้งละหนึ่งห้อง ถ้ามีการจองเดิมจะถูกแทนที่) date รูปแบบ YYYY-MM-DD, time รูปแบบ HH:MM"""
    if time not in FREE.get(date, {}).get(room, []):
        return {"ok": False, "error": f"{room} วันที่ {date} เวลา {time} ไม่ว่าง"}
    BOOKING.clear(); BOOKING.update(room=room, date=date, time=time)
    return {"ok": True, "booking": dict(BOOKING)}

In [ ]:
TOOLS = [get_calendar, find_rooms, book_room]
print(book_room.name)
print(book_room.args)

## 3. Agent loop: `create_agent`

ใน Lab 1: `run_agent()` ราว 20 บรรทัด → ใน LangChain: ฟังก์ชันเดียว

In [ ]:
# ✏️ TODO 2 (▶ วิดีโอ 5:47): สร้าง agent จาก model, tool ทั้งสามตัว และ system prompt
from langchain.agents import create_agent

agent = create_agent(llm, tools=___, system_prompt=SYSTEM)

In [ ]:
# ---- ช่วยพิมพ์: แสดงเฉพาะข้อความของรอบล่าสุด (เหมือน print ในลูปของ Lab 1) ----
def show(result):
    msgs = result["messages"]
    last_user = max(i for i, m in enumerate(msgs) if m.type == "human")
    for m in msgs[last_user + 1:]:
        for c in getattr(m, "tool_calls", []):
            print(f"  🔧 {c['name']}({c['args']})")
        if m.type == "tool":
            print(f"  👀 {m.content}")
    if "__interrupt__" in result:
        for r in result["__interrupt__"][0].value["action_requests"]:
            print(f"⏸ รออนุมัติ: {r['name']}({r['args']})")
    else:
        print(msgs[-1].content)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})
show(result)

In [ ]:
print([m.type for m in result["messages"]])

## 4. Persistence: `checkpointer` + `thread_id`

ใน Lab 1 เราคุยต่อได้เพราะส่ง `history` list เดิมเข้าไปเอง — แล้ว `invoke` ครั้งที่สองล่ะ?

In [ ]:
show(agent.invoke({"messages": [{"role": "user", "content": "13:00 ครับ"}]}))
print(BOOKING)

In [ ]:
# ✏️ TODO 3 (▶ วิดีโอ 8:31): ให้ agent จำประวัติได้ ด้วย checkpointer
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
agent = create_agent(llm, tools=TOOLS, system_prompt=SYSTEM, checkpointer=___)

config = {"configurable": {"thread_id": "student-1"}}

In [ ]:
show(agent.invoke({"messages": [{"role": "user", "content": QUESTION}]}, config))

In [ ]:
# ✏️ TODO 4 (▶ วิดีโอ 9:28): คุยต่อใน thread เดิม: ส่ง config ไปกับ invoke
show(agent.invoke({"messages": [{"role": "user", "content": "13:00 ครับ"}]}, ___))
print(BOOKING)

In [ ]:
show(agent.invoke({"messages": [{"role": "user", "content": "ขอเลื่อนเป็นบ่ายสามได้ไหม"}]},
                  {"configurable": {"thread_id": "student-2"}}))

## 5. Safety: middleware

**middleware** คือโค้ดที่เสียบเข้าไปในลูปของ `create_agent` — ก่อน/หลังเรียก model หรือก่อนรัน tool

| Lab 1 | LangChain |
|---|---|
| `MAX_STEPS` | `ModelCallLimitMiddleware(run_limit=...)` |
| `NEEDS_APPROVAL` + `PENDING` | `HumanInTheLoopMiddleware(interrupt_on={...})` |
| `approve(history, ok=True)` | `invoke(Command(resume={"decisions": [{"type": "approve"}]}), config)` |

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, HumanInTheLoopMiddleware
from langgraph.types import Command

limited = create_agent(llm, tools=TOOLS, system_prompt=SYSTEM,
                       middleware=[ModelCallLimitMiddleware(run_limit=1)])
show(limited.invoke({"messages": [{"role": "user", "content": QUESTION}]}))

In [ ]:
# ✏️ TODO 5 (▶ วิดีโอ 11:46): ให้ book_room ต้องรอคนอนุมัติก่อนรัน
safe = create_agent(llm, tools=TOOLS, system_prompt=SYSTEM, checkpointer=memory,
                    middleware=[HumanInTheLoopMiddleware(interrupt_on=___)])

In [ ]:
show(safe.invoke({"messages": [{"role": "user", "content": "ขอเลื่อนเป็นบ่ายสามได้ไหม"}]}, config))
print(BOOKING)

In [ ]:
# ✏️ TODO 6 (▶ วิดีโอ 12:48): อนุมัติคำขอที่รออยู่ แล้วให้ agent ทำต่อ
show(safe.invoke(Command(resume={"decisions": [{"type": ___}]}), config))
print(BOOKING)

## สรุป: เขียนเอง vs framework

| | Lab 1 (เขียนเอง) | Lab 2 (LangChain) |
|---|---|---|
| เรียก model | `client` + `call_llm` | `ChatOpenAI` |
| tool | ฟังก์ชัน + schema + `FUNCS` + `run_tool` | `@tool` |
| agent loop | `run_agent()` | `create_agent()` |
| ความจำ | `history` list | `checkpointer` + `thread_id` |
| จำกัดรอบ | `MAX_STEPS` | `ModelCallLimitMiddleware` |
| ให้คนอนุมัติ | `NEEDS_APPROVAL` + `approve()` | `HumanInTheLoopMiddleware` + `Command(resume=...)` |
| ตรวจห้องว่างก่อนจอง | ใน `book_room` | ใน `book_room` (ยังเป็นงานของเรา) |

framework ช่วยให้โค้ดสั้นลงและมีของพร้อมใช้ (เก็บประวัติลงฐานข้อมูล, reject/edit) — แต่ทุกชิ้นคือสิ่งที่เราเขียนเองได้ใน Lab 1
เมื่อ agent ทำตัวแปลกๆ ความเข้าใจจาก Lab 1 คือสิ่งที่ช่วยให้รู้ว่าต้องดูตรงไหน

**ลองทำต่อ:** ลองไม่อนุมัติ `{"type": "reject", "message": "ขอเป็นวันศุกร์แทน"}` แล้วดูว่า agent ทำอะไรต่อ ·
เพิ่ม tool `cancel_booking` แล้วใส่ใน `interrupt_on`

**คลิปถัดไป — สรุป EP03:** ทบทวน agent loop, harness และ 5 ความท้าทาย จากสิ่งที่เราเห็นในสอง lab